# Final Modeling 

A neat compilation of initial model & improvements. 

In [1]:
import pandas as pd
import numpy as np

from linearmodels import PanelOLS

In [2]:
# this data has already been cleaned from a different file, see "Exported_Cleaned_Data.ipynb" and "data_cleaning.py" for further details
# this data contains poverty values for baseline year 2019, which is necessary for modeling
grade_data = pd.read_csv('cleaned_school_data_updated.csv')

In [3]:
# fix poverty values
grade_data['poverty_percentage'] = (grade_data['% Poverty'] * 100).round(3)

In [4]:
grade_data_clean = grade_data.dropna(subset=['% Poverty', 'mean_scale_score'])

## Individual Grade Datasets

In [5]:
# grade splits

data3 = grade_data[
    (grade_data['Grade'] == '3') & 
    (grade_data['Student Category'] == 'All Students') & 
    (grade_data['Report Category'] == 'School')
]


data4 = grade_data[
    (grade_data['Grade'] == '4') &  
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data5 = grade_data[
    (grade_data['Grade'] == '5') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data6 = grade_data[
    (grade_data['Grade'] == '6') &   
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data7 = grade_data[
    (grade_data['Grade'] == '7') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data8 = grade_data[
    (grade_data['Grade'] == '8') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]



## Model V1: Binary Poverty Variable

In [6]:
# merged_data_titlei = merge_data_and_poverty(school_data_3, school_poverty)

# fix poverty percentage to float, categorize, round to 3 decimals
from data_cleaning import categorize_title_i


grade_data_clean['% Poverty'] = grade_data_clean['% Poverty'].astype(float)
grade_data_clean['Title I Category'] = grade_data_clean.apply(categorize_title_i, axis=1)
# grade_data_clean['% Poverty'] = grade_data_clean['% Poverty'].round(3)


C:\Users\madis\AppData\Local\Temp\ipykernel_35664\2707232197.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grade_data_clean['% Poverty'] = grade_data_clean['% Poverty'].astype(float)
C:\Users\madis\AppData\Local\Temp\ipykernel_35664\2707232197.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grade_data_clean['Title I Category'] = grade_data_clean.apply(categorize_title_i, axis=1)


In [7]:
# model v1 only uses years 2019 and 2022 
modelv1_data = grade_data_clean[
    (grade_data_clean['Year'].isin([2019, 2022])) &
    (grade_data_clean['Student Category'] == 'All Students') &  
    (grade_data_clean['Report Category'] == 'School')
]

In [8]:
# set index for PanelOLS
data_titlei= modelv1_data.set_index(['school_name','Year'])

In [9]:
# setting up for interaction term for did model
data_titlei['Title I Category'] = data_titlei['Title I Category'].astype(int)
# data_titlei['treated'] = (data_titlei['Title I Category'] == 1).astype(int)
data_titlei['post'] = (data_titlei.index.get_level_values('Year') == 2022).astype(int)
data_titlei['did_coef'] = (data_titlei['Title I Category'] * data_titlei['post']).astype(int)

In [10]:
def run_did_model(data):
    model = PanelOLS.from_formula('mean_scale_score ~ did_coef + EntityEffects + TimeEffects', data=data, weights=data['number_tested'])
    results = model.fit(cov_type='clustered', cluster_entity=True)
    return results.summary

In [11]:
# loop through all grades and run DiD models for each grade level
for grade in ['3', '4', '5', '6', '7', '8']:
    grade_data = data_titlei[data_titlei['Grade'] == grade]
    if not grade_data.empty:
        summary = run_did_model(grade_data)
        print(f"Grade {grade} DiD Model Summary:")
        print(summary)
       


Grade 3 DiD Model Summary:
                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0369
Estimator:                   PanelOLS   R-squared (Between):             -0.0027
No. Observations:                1550   R-squared (Within):               0.0673
Date:                Wed, May 06 2026   R-squared (Overall):             -0.0027
Time:                        12:43:12   Log-likelihood                   -3653.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      29.393
Entities:                         780   P-value                           0.0000
Avg Obs:                       1.9872   Distribution:                   F(1,768)
Min Obs:                       1.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             15.802
 

## Working Example of Placebo Model for Parallel Trends

In [13]:
grade_data.reset_index(inplace=True)

In [14]:
grade_data_clean = grade_data.dropna(subset=['% Poverty', 'mean_scale_score'])

In [15]:
# Time-placebo test: use 2018 and 2019 as "pre" period
# this is a method to test the parallel trends assumption by checking for any pre-existing trends in the outcome variable before the treatment period
pre_data = grade_data_clean[grade_data_clean['Year'].isin([2018,2019])].copy() # new pre data
pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int) # treating 2019 as the "post" period in the placebo test
pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage'] # new placebo interaction term for treated

In [16]:
pre_data_test = pre_data[pre_data['Report Category'] == 'School'].copy() # only school, no district or citywide
pre_data_test = pre_data_test.set_index(['DBN', 'Year']) # set index for PanelOLS

In [17]:
# loop through all datasets and run placebo DiD models to test parallel trends assumption
results = {}
for i in range(3, 9):
    data = globals()[f"data{i}"]
    
    pre_data = data[data['Year'].isin([2018, 2019])].copy()
    pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int)
    pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage']
    
    pre_data = pre_data.set_index(['DBN', 'Year'])
    
    model = PanelOLS(
        dependent=pre_data['mean_scale_score'],
        exog=pre_data[['fake_treated_cont']],
        entity_effects=True,
        time_effects=True,
        weights=pre_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    results[i] = model
    
    print(f"Dataset {i} → Coef: {model.params['fake_treated_cont']:.4f}, "
          f"p-value: {model.pvalues['fake_treated_cont']:.4f}")

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 3 → Coef: 0.0230, p-value: 0.0182
Dataset 4 → Coef: 0.0289, p-value: 0.0036
Dataset 5 → Coef: -0.0058, p-value: 0.5272


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 6 → Coef: -0.0395, p-value: 0.0011
Dataset 7 → Coef: -0.0396, p-value: 0.0040
Dataset 8 → Coef: -0.0007, p-value: 0.9570


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


## Model For Each Grade - Mean Scale Score

#### Verify Index

In [22]:
improved_model_data = pd.read_csv('cleaned_school_data_updated.csv')

In [36]:
# fix poverty values
improved_model_data['poverty_percentage'] = (improved_model_data['% Poverty'] * 100).round(3)

In [43]:
improved_model_data = improved_model_data.dropna(subset=['% Poverty', 'mean_scale_score'])

In [44]:
# grade splits

data3_imp = improved_model_data[
    (improved_model_data['Grade'] == '3') & 
    (improved_model_data['Student Category'] == 'All Students') & 
    (improved_model_data['Report Category'] == 'School')
]


data4_imp = improved_model_data[
    (improved_model_data['Grade'] == '4') &  
    (improved_model_data['Student Category'] == 'All Students') &
    (improved_model_data['Report Category'] == 'School')
]

data5_imp = improved_model_data[
    (improved_model_data['Grade'] == '5') &
    (improved_model_data['Student Category'] == 'All Students') &
    (improved_model_data['Report Category'] == 'School')
]

data6_imp = improved_model_data[
    (improved_model_data['Grade'] == '6') &   
    (improved_model_data['Student Category'] == 'All Students') &
    (improved_model_data['Report Category'] == 'School')
]

data7_imp = improved_model_data[
    (improved_model_data['Grade'] == '7') &
    (improved_model_data['Student Category'] == 'All Students') &
    (improved_model_data['Report Category'] == 'School')
]

data8_imp = improved_model_data[
    (improved_model_data['Grade'] == '8') &
    (improved_model_data['Student Category'] == 'All Students') &
    (improved_model_data['Report Category'] == 'School')
]



In [45]:
def verify_index(df):

    ''' This function verifies that each school (DBN) has at least 3 years of data for the PanelOLS. Removes
    schools that do not include all three years.
    
    Input: 
        df: dataframe with indexed DBN and Year
    Output:
        df_filtered: filtered data that contains all schools for each grade with at all three years of data
    '''
    
    # count number of years in the data
    year_counts = df.reset_index().groupby("DBN")["Year"].count()
    # schools to keep are those with three years of data
    schools_to_keep = year_counts[year_counts >= 3].index
    # filter the data to keep only those schools
    
    df_filtered = df[df.index.get_level_values('DBN').isin(schools_to_keep)]
   
    return df_filtered

In [46]:
data3_imp = data3_imp.set_index(['DBN', 'Year'])
data4_imp = data4_imp.set_index(['DBN', 'Year'])
data5_imp = data5_imp.set_index(['DBN', 'Year'])
data6_imp = data6_imp.set_index(['DBN', 'Year'])
data7_imp = data7_imp.set_index(['DBN', 'Year'])
data8_imp = data8_imp.set_index(['DBN', 'Year'])

In [47]:
data3_imp = verify_index(data3_imp)
data4_imp = verify_index(data4_imp) 
data5_imp = verify_index(data5_imp)
data6_imp = verify_index(data6_imp)
data7_imp = verify_index(data7_imp)
data8_imp = verify_index(data8_imp)

In [48]:

def run_grade_model(data):

    grade_data = data
    
    # 2019 poverty rate as baseline for each school
    poverty_baseline = grade_data[grade_data['Year'] == 2019][
        ['DBN', 'poverty_percentage']
    ].rename(columns={'poverty_percentage': 'poverty_baseline'})
    
    # merge the baseline poverty rate back into the main data
    grade_data = grade_data.merge(poverty_baseline, on='DBN', how='left')
    
    # interaction terms using new poverty baseline(2019) for pre and post periods
    grade_data['poverty_pre'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2018).astype(int)
    )
    grade_data['poverty_post'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2022).astype(int)
    )
    
    # index for PanelOLS
    grade_data = grade_data.set_index(['DBN', 'Year'])
    
    # PanelOLS model with clustered standard errors at the school level,
    # controlling for school and year fixed effects, 
    # and weighted by number of students tested
    model = PanelOLS(
        dependent=grade_data['mean_scale_score'],
        exog=grade_data[['poverty_pre', 'poverty_post']],
        entity_effects=True,
        time_effects=True,
        weights=grade_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    return model



In [ ]:
# run models for each grade level
data3_imp.reset_index(inplace=True)
model_3 = run_grade_model(data3_imp)

data4_imp.reset_index(inplace=True)
model_4 = run_grade_model(data4_imp)

data5_imp.reset_index(inplace=True)
model_5 = run_grade_model(data5_imp)

data6_imp.reset_index(inplace=True)
model_6 = run_grade_model(data6_imp)

data7_imp.reset_index(inplace=True)
model_7 = run_grade_model(data7_imp)

data8_imp.reset_index(inplace=True)
model_8 = run_grade_model(data8_imp)

In [50]:
model_3

Dep. Variable:,mean_scale_score,R-squared:,0.0209
Estimator:,PanelOLS,R-squared (Between):,-0.0051
No. Observations:,2274,R-squared (Within):,-0.0483
Date:,"Wed, May 06 2026",R-squared (Overall):,-0.0051
Time:,13:00:29,Log-likelihood,-5549.1
Cov. Estimator:,Clustered,,
,,F-statistic:,16.125
Entities:,758,P-value,0.0000
Avg Obs:,3.0000,Distribution:,"F(2,1512)"
Min Obs:,3.0000,,
Max Obs:,3.0000,F-statistic (robust):,10.647


In [51]:
model_4

Dep. Variable:,mean_scale_score,R-squared:,0.0425
Estimator:,PanelOLS,R-squared (Between):,-0.0072
No. Observations:,2226,R-squared (Within):,0.1103
Date:,"Wed, May 06 2026",R-squared (Overall):,-0.0072
Time:,13:00:29,Log-likelihood,-5502.0
Cov. Estimator:,Clustered,,
,,F-statistic:,32.828
Entities:,742,P-value,0.0000
Avg Obs:,3.0000,Distribution:,"F(2,1480)"
Min Obs:,3.0000,,
Max Obs:,3.0000,F-statistic (robust):,20.269


In [53]:
model_5

Dep. Variable:,mean_scale_score,R-squared:,0.0051
Estimator:,PanelOLS,R-squared (Between):,0.0019
No. Observations:,2211,R-squared (Within):,0.0163
Date:,"Wed, May 06 2026",R-squared (Overall):,0.0019
Time:,13:00:30,Log-likelihood,-5198.1
Cov. Estimator:,Clustered,,
,,F-statistic:,3.8017
Entities:,737,P-value,0.0226
Avg Obs:,3.0000,Distribution:,"F(2,1470)"
Min Obs:,3.0000,,
Max Obs:,3.0000,F-statistic (robust):,2.5634


In [54]:
model_6

Dep. Variable:,mean_scale_score,R-squared:,0.0952
Estimator:,PanelOLS,R-squared (Between):,0.0104
No. Observations:,1392,R-squared (Within):,0.0125
Date:,"Wed, May 06 2026",R-squared (Overall):,0.0104
Time:,13:00:30,Log-likelihood,-3189.7
Cov. Estimator:,Clustered,,
,,F-statistic:,48.634
Entities:,464,P-value,0.0000
Avg Obs:,3.0000,Distribution:,"F(2,924)"
Min Obs:,3.0000,,
Max Obs:,3.0000,F-statistic (robust):,18.545


In [55]:
model_7

Dep. Variable:,mean_scale_score,R-squared:,0.0981
Estimator:,PanelOLS,R-squared (Between):,0.0103
No. Observations:,1368,R-squared (Within):,0.3254
Date:,"Wed, May 06 2026",R-squared (Overall):,0.0103
Time:,13:00:30,Log-likelihood,-2981.1
Cov. Estimator:,Clustered,,
,,F-statistic:,49.410
Entities:,456,P-value,0.0000
Avg Obs:,3.0000,Distribution:,"F(2,908)"
Min Obs:,3.0000,,
Max Obs:,3.0000,F-statistic (robust):,13.577


In [56]:
model_8

Dep. Variable:,mean_scale_score,R-squared:,0.0005
Estimator:,PanelOLS,R-squared (Between):,0.0002
No. Observations:,1362,R-squared (Within):,-0.0159
Date:,"Wed, May 06 2026",R-squared (Overall):,0.0002
Time:,13:00:30,Log-likelihood,-2894.1
Cov. Estimator:,Clustered,,
,,F-statistic:,0.2472
Entities:,454,P-value,0.7810
Avg Obs:,3.0000,Distribution:,"F(2,904)"
Min Obs:,3.0000,,
Max Obs:,3.0000,F-statistic (robust):,0.0820


### HonestDiD

In [57]:
import os

In [58]:
os.makedirs("honestdid_coefs_complete", exist_ok=True)

In [59]:
def export_coefficients2(model, grade):
    ''' This function exports the coefficients and variance-covariance matrix of PanelOLS models to be imported
    into R for HonestDiD analysis.
    
    Input:
        model: PanelOLS model
        grade: grade level for naming the output files
    Output:
        two csv files containing coefficients and variance-covariance matrix for the model.
        '''
    
    np.savetxt(f"honestdid_coefs_complete/beta_grade{grade}.csv",
               model.params.values, delimiter=",")
    np.savetxt(f"honestdid_coefs_complete/vcov_grade{grade}.csv",
               model.cov.values, delimiter=",")

In [ ]:
# export_coefficients2(model_3, 3)
# export_coefficients2(model_4, 4)
# export_coefficients2(model_5, 5)
# export_coefficients2(model_6, 6)
# export_coefficients2(model_7, 7)
# export_coefficients2(model_8, 8)